In [1]:
import pyspark.sql.functions as F
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

In [2]:
spark = configure_spark_with_delta_pip(
    SparkSession.builder.appName("Fligh Delays and Cancellations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 17:17:28 WARN Utils: Your hostname, humacao, resolves to a loopback address: 127.0.1.1; using 192.168.68.83 instead (on interface wlp2s0)
26/03/15 17:17:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/sparks/spark-4.1.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/daniel/.ivy2.5.2/cache
The jars for the packages stored in: /home/daniel/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-de922ff4-1365-4d57-b3cc-2886dbcd1923;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4

# Read Data Files

In [3]:
DIRECTORY = "./dataset"

In [4]:
airlines = spark.read.csv(f"{DIRECTORY}/airlines.csv", header=True, inferSchema=True)

In [5]:
airports = spark.read.csv(f"{DIRECTORY}/airports.csv", header=True, inferSchema=True)

In [6]:
flights = spark.read.csv(f"{DIRECTORY}/flights.csv", header=True, inferSchema=True)

Explore Inferred Schema and Create Explicit Schema (if needed)

In [7]:
airlines.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)



In [8]:
airlines.show(5, False)

+---------+----------------------+
|IATA_CODE|AIRLINE               |
+---------+----------------------+
|UA       |United Air Lines Inc. |
|AA       |American Airlines Inc.|
|US       |US Airways Inc.       |
|F9       |Frontier Airlines Inc.|
|B6       |JetBlue Airways       |
+---------+----------------------+
only showing top 5 rows


In [9]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [10]:
airports.show(5, False)

+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY       |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown  |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene    |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque|NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen   |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany     |GA   |USA    |31.53552|-84.19447 |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows


In [11]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [12]:
flights.show(5, False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

26/03/15 17:17:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


# Save Raw Data to Delta Table Bronze Layer

In [15]:
(
    airlines.write.format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airlines_bronze")
)

In [16]:
(
    airports.write.format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airports_bronze")
)

In [17]:
(
    flights.write.format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/flights_bronze")
)

26/03/15 17:19:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

# Read Delta Tables from Bronze Layer

In [18]:
airlines_bronze = spark.read.format("delta").load("./medallion/bronze/airlines_bronze")

In [19]:
airports_bronze = spark.read.format("delta").load("./medallion/bronze/airports_bronze")

In [20]:
flights_bronze = spark.read.format("delta").load("./medallion/bronze/flights_bronze")

# Data Checks

In [21]:
airlines_bronze.show(14, False)

+---------+----------------------------+
|IATA_CODE|AIRLINE                     |
+---------+----------------------------+
|UA       |United Air Lines Inc.       |
|AA       |American Airlines Inc.      |
|US       |US Airways Inc.             |
|F9       |Frontier Airlines Inc.      |
|B6       |JetBlue Airways             |
|OO       |Skywest Airlines Inc.       |
|AS       |Alaska Airlines Inc.        |
|NK       |Spirit Air Lines            |
|WN       |Southwest Airlines Co.      |
|DL       |Delta Air Lines Inc.        |
|EV       |Atlantic Southeast Airlines |
|HA       |Hawaiian Airlines Inc.      |
|MQ       |American Eagle Airlines Inc.|
|VX       |Virgin America              |
+---------+----------------------------+



In [22]:
airlines_bronze.count()

14

In [50]:
airlines_bronze.describe()

DataFrame[summary: string, IATA_CODE: string, AIRLINE: string]

In [52]:
airlines_bronze.summary()

DataFrame[summary: string, IATA_CODE: string, AIRLINE: string]

In [53]:
airports_bronze.count()

322

In [54]:
airports_bronze.show(10, False)

+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown    |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene      |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque  |NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen     |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany       |GA   |USA    |31.53552|-84.19447 |
|ACK      |Nantucket Memorial Airport         |Nantucket    |MA   |USA    |41.25305|-70.06018 |
|ACT      |Waco Regional Airport              |Waco         |TX   |USA    |31.61129|-97.23052 |
|ACV      |Arcata Airport               

In [55]:
airports_bronze.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [56]:
airports_bronze.describe()

DataFrame[summary: string, IATA_CODE: string, AIRPORT: string, CITY: string, STATE: string, COUNTRY: string, LATITUDE: string, LONGITUDE: string]

In [57]:
airports_bronze.summary()

DataFrame[summary: string, IATA_CODE: string, AIRPORT: string, CITY: string, STATE: string, COUNTRY: string, LATITUDE: string, LONGITUDE: string]

In [59]:
flights_bronze.show(truncate=False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

In [60]:
flights_bronze.count()

5819079

In [61]:
flights_bronze.describe()

DataFrame[summary: string, YEAR: string, MONTH: string, DAY: string, DAY_OF_WEEK: string, AIRLINE: string, FLIGHT_NUMBER: string, TAIL_NUMBER: string, ORIGIN_AIRPORT: string, DESTINATION_AIRPORT: string, SCHEDULED_DEPARTURE: string, DEPARTURE_TIME: string, DEPARTURE_DELAY: string, TAXI_OUT: string, WHEELS_OFF: string, SCHEDULED_TIME: string, ELAPSED_TIME: string, AIR_TIME: string, DISTANCE: string, WHEELS_ON: string, TAXI_IN: string, SCHEDULED_ARRIVAL: string, ARRIVAL_TIME: string, ARRIVAL_DELAY: string, DIVERTED: string, CANCELLED: string, CANCELLATION_REASON: string, AIR_SYSTEM_DELAY: string, SECURITY_DELAY: string, AIRLINE_DELAY: string, LATE_AIRCRAFT_DELAY: string, WEATHER_DELAY: string]

In [62]:
flights_bronze.summary()

DataFrame[summary: string, YEAR: string, MONTH: string, DAY: string, DAY_OF_WEEK: string, AIRLINE: string, FLIGHT_NUMBER: string, TAIL_NUMBER: string, ORIGIN_AIRPORT: string, DESTINATION_AIRPORT: string, SCHEDULED_DEPARTURE: string, DEPARTURE_TIME: string, DEPARTURE_DELAY: string, TAXI_OUT: string, WHEELS_OFF: string, SCHEDULED_TIME: string, ELAPSED_TIME: string, AIR_TIME: string, DISTANCE: string, WHEELS_ON: string, TAXI_IN: string, SCHEDULED_ARRIVAL: string, ARRIVAL_TIME: string, ARRIVAL_DELAY: string, DIVERTED: string, CANCELLED: string, CANCELLATION_REASON: string, AIR_SYSTEM_DELAY: string, SECURITY_DELAY: string, AIRLINE_DELAY: string, LATE_AIRCRAFT_DELAY: string, WEATHER_DELAY: string]

## Check foir `null` values

### Airports DataFrame

In [27]:
[
    airports_bronze.filter(F.col(c).isNull()).show(truncate=False)
    for c in airports_bronze.columns
]

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE|AIRPORT|CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+-------+----+-----+-------+--------+---------+
+---------+-------+----+-----+-------+--------+---------+

+---------+-------+----+-----+-------+--------+---------+
|IATA_CODE

[None, None, None, None, None, None, None]

There are 3 airports that have no Lat Lon data. 

In [28]:
airports_bronze.filter(F.col("LATITUDE").isNull()).show(truncate=False)

+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+
|IATA_CODE|AIRPORT                                                   |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+
|ECP      |Northwest Florida Beaches International Airport           |Panama City  |FL   |USA    |NULL    |NULL     |
|PBG      |Plattsburgh International Airport                         |Plattsburgh  |NY   |USA    |NULL    |NULL     |
|UST      |Northeast Florida Regional Airport (St. Augustine Airport)|St. Augustine|FL   |USA    |NULL    |NULL     |
+---------+----------------------------------------------------------+-------------+-----+-------+--------+---------+



Get airport code for those airports to check if those airports are in flights data as destination or origin airports.

In [29]:
null_airports = airports_bronze.filter(F.col("LATITUDE").isNull()).select("IATA_CODE")

In [30]:
null_airport_codes = [row.IATA_CODE for row in null_airports.collect()]

In [31]:
print(null_airport_codes)

['ECP', 'PBG', 'UST']


In [32]:
flights_bronze.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

Check if airport codes are in flight data.

In [33]:
flights_bronze.filter(
    F.col("DESTINATION_AIRPORT").isin(null_airport_codes)
    | F.col("ORIGIN_AIRPORT").isin(null_airport_codes)
).count()

9215

Since they are, try to get lat lon data from other sources and fix airports table for Silver layer.

In [34]:
airports_bronze.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [35]:
print(null_airport_codes)

['ECP', 'PBG', 'UST']


Manually update lat and lon for airports.

In [36]:
aisports_silver = (
    airports_bronze.withColumn(
        "LATITUDE",
        F.when(F.col("IATA_CODE") == "ECP", 30.3549).otherwise(F.col("LATITUDE")),
    )
    .withColumn(
        "LONGITUDE",
        F.when(F.col("IATA_CODE") == "ECP", -85.7995).otherwise(F.col("LONGITUDE")),
    )
    .withColumn(
        "LATITUDE",
        F.when(F.col("IATA_CODE") == "PBG", 44.6505).otherwise(F.col("LATITUDE")),
    )
    .withColumn(
        "LONGITUDE",
        F.when(F.col("IATA_CODE") == "PBG", -73.4675).otherwise(F.col("LONGITUDE")),
    )
    .withColumn(
        "LATITUDE",
        F.when(F.col("IATA_CODE") == "UST", 29.9555).otherwise(F.col("LATITUDE")),
    )
    .withColumn(
        "LONGITUDE",
        F.when(F.col("IATA_CODE") == "UST", -81.3372).otherwise(F.col("LONGITUDE")),
    )
)

Recheck data for nulls

In [37]:
[aisports_silver.filter(F.col(c).isNull()).count() for c in aisports_silver.columns]

[0, 0, 0, 0, 0, 0, 0]

Save cleaned dataframe to Silver layer.

In [38]:
(
    aisports_silver.write.format("delta")
    .mode("overwrite")
    .save("./medallion/silver/airports_silver")
)

### Airlines DataFrame

In [39]:
[airlines_bronze.filter(F.col(c).isNull()).count() for c in airlines_bronze.columns]

[0, 0]

Save arilines DataFrame to Silver layer (There were no changes to tis data. It is the same as the Bronze layer file.)

In [40]:
(
    airlines_bronze.write.format("delta")
    .mode("overwrite")
    .save("./medallion/silver/airlines_silver")
)

### Flights DataFrame

In [42]:
flights_bronze.count()

5819079

In [45]:
flights_bronze.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [46]:
[flights_bronze.filter(F.col(c).isNull()).count() for c in flights_bronze.columns]

[0,
 0,
 0,
 0,
 0,
 0,
 14721,
 0,
 0,
 0,
 86153,
 86153,
 89047,
 89047,
 6,
 105071,
 105071,
 0,
 92513,
 92513,
 0,
 92513,
 105071,
 0,
 0,
 5729195,
 4755640,
 4755640,
 4755640,
 4755640,
 4755640]

In [47]:
(flights_bronze.filter(F.col("TAIL_NUMBER").isNull()).show())

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-